# LLM functions and chat objects

Anton Antonov   
["Jupyter::Chatbook" Raku package at GitHub](https://github.com/antononcube/Raku-Jupyter-Chatbook)   
August, September 2023

------

## Introduction

In this notebook we show how Large Language Models (LLMs) functions and LLM chat objects can be created and used in a notebook. This notebook is a ***chatbook*** created with the Raku package ["Jupyter::Chatbook"](https://raku.land/zef:antononcube/Jupyter::Chatbook), [AAp1].

A "streamlined" way to utilize LLMs is with the package["LLM::Functions"](https://raku.land/zef:antononcube/LLM::Functions), [AA1, AAp2]. I.e. without using specific, dedicated packages for accessing LLMs like those of ["WWW::OpenAI"](https://raku.land/zef:antononcube/WWW::OpenAI), [AAp3], and ["WWW::PaLM"](https://raku.land/zef:antononcube/WWW::PaLM), [AAp4].

Chatbooks load in their initialization phase the package 
["LLM::Functions"](https://github.com/antononcube/Raku-LLM-Functions), [AAp2].
Also, in the initialization phase are loaded the packages 
["Clipboard"](https://github.com/antononcube/Raku-Clipboard), [AAp5],
["Data::Translators"](https://github.com/antononcube/Raku-Data-Translators), [AAp6],
["Data::TypeSystem"](https://github.com/antononcube/Raku-Data-Translators), [AAp7],
["Text::Plot"](https://github.com/antononcube/Raku-Text-Plot), [AAp8],
and 
["Text::SubParsers"](https://github.com/antononcube/Raku-Text-SubParsers), [AAp9], 
that can be used to post-process LLM outputs.

**Remark:** For LLM functions and chat objects the functionalities of ["Jupyter::Chatbook"](https://github.com/bduggan/raku-jupyter-kernel), [BDp1], suffice. In other words, the "chatbook" extensions provided by "Jupyter::Chatbook", [AAp1], are not needed.

**Remark:** The LLM functions and chat objects are provided by the package ["LLM::Functions"](https://raku.land/zef:antononcube/LLM::Functions), [AA1, AAp2], which in turn uses the packages ["WWW::OpenAI"](https://raku.land/zef:antononcube/WWW::OpenAI), [AAp3], and ["WWW::PaLM"](https://raku.land/zef:antononcube/WWW::PaLM), [AAp4].

**Remark:** The default API keys for the LLM functions and chat objects are taken from the Operating System (OS) environmental variables `OPENAI_API_KEY` and `PALM_API_KEY`. The api keys can also be specified using LLM evaluator and configuration options and objects; see [AA1, AAp2].


------ 

## LLM functions



Here is an example of an LLM function that takes 3 arguments:

In [1]:
my &f1 = llm-function({"What is the $^a of $^b in $^c. Give numerical answer only."})

LLM::Function(-> **@args, *%args { #`(Block|5464178980840) ... }, 'chatgpt')

Here the LLM function defined above is "concretized" with argument values:

In [2]:
&f1('GDP', 'California, USA', '2020')

3135000000000

Here we call repeatedly `&f1` in order to Lake Mead levels: 

In [3]:
'level' X 'Lake Mead' X (1990...2010) ==> map({ $_.tail => &f1(|$_) }) ==> my @levels;

(1990 => 1120 1991 => 1075 1992 => 1100 1993 => 1,211 feet 1994 => 1203 1995 => 1092 1996 => 1100 1997 => 1201 1998 => 1,211 feet 1999 => 1192 2000 => 1,211 feet 2001 => 1083 2002 => 1,081 feet 2003 => 1114 2004 => 1100 2005 => 1085 2006 => 1104 2007 => 1125 2008 => 1091 2009 => 1075 2010 => 1,117 feet)

Here we extract numbers of the obtained LLM results and plot the levels over the corresponding years:

In [4]:
@levels.map({ ($_.key, sub-parser('GeneralNumber', :drop).process($_.value).head) }) ==> text-list-plot(width=>60)

+---+------------+-----------+------------+------------+---+         
|                                                          |         
+          *  *       * *    *                             +  1200.00
|                          *                               |         
|                                                          |         
|                                                          |         
+                                                          +  1150.00
|                                                          |         
|                                               *          |         
|   *                                *                 *   |         
+        *         *                    *    *             +  1100.00
|                *                                *        |         
|     *                         *  *      *          *     |         
|                                                          |         
+---+------------+--

Here we defined another LLM function, and since the LLM function produces (or it is suppossed to produce) HTML code we use the magic spec `%% html`:

In [5]:
%% html

my &ftbl = llm-function({"The HTML code of a random table with $^a rows and $^b columns is : "}, e => 'ChatGPT');

&ftbl(4,3)

Header 1,Header 2,Header 3
"Row 1, Cell 1","Row 1, Cell 2","Row 1, Cell 3"
"Row 2, Cell 1","Row 2, Cell 2","Row 2, Cell 3"
"Row 3, Cell 1","Row 3, Cell 2","Row 3, Cell 3"


Detailed examples of LLM workflows can be found in the article ["Workflows with LLM functions"](https://rakuforprediction.wordpress.com/2023/08/01/workflows-with-llm-functions/), [AA1].

------

## Number extraction from LLM responses

Often LLM return larger number with comma delimiters between the digits. Here is an example: 

In [8]:
my $pop = llm-function()("What is the population of Niger?")

As of the most recent estimates in 2023, the population of Niger is approximately 26 million people. Keep in mind that population figures can vary slightly depending on the source and the year of the estimate.

One way to extract the numbers from those responses is to use the token `<local-number>` provided by the package ["Intl::Token::Number"](https://raku.land/zef:guifa/Intl::Token::Number), [MSp1]:

In [9]:
use Intl::Token::Number;

$pop ~~ m:g/ <local-number> /

(｢2023｣
 local-number => ｢2023｣ ｢26｣
 local-number => ｢26｣)

Alternatively, a sub-parser from the packatge ["Text::SubParsers"](https://raku.land/zef:antononcube/Text::SubParsers), [AAp8], can be used:

In [10]:
sub-parser(Whatever).subparse($pop).raku

$["As of the most recent estimates in", 2023, ", the population of Niger is approximately", 26, "million people. Keep in mind that population figures can vary slightly depending on the source and the year of the estimate."]

-------

## LLM chat objects

LLM chat objects are used to provide context for LLM interactions with different messages.
LLM chat objects keep track of the conversation history, so LLM can better understand the meaning of each message.

For more involved examples of using LLM chat objects see the article ["Number guessing games: PaLM vs ChatGPT"](https://rakuforprediction.wordpress.com/2023/08/06/number-guessing-games-palm-vs-chatgpt/), [AA2].


Make a chat object:

In [11]:
my $chatObj = llm-chat(
    conf=>'OpenAI',
    prompt => 'You are Raku coding instructor. You are the best Raku programmer and you know Raku documentation very well. You answers are concise, mostly with code');

LLM::Functions::Chat(chat-id = , llm-evaluator.conf.name = openai, messages.elems = 0)

Here we give the chat object a message, and specify the output to be in Markdown format using the magic spec `%% markdown`:

In [12]:
%% markdown
$chatObj.eval('How do you get the characters of a string?')

[(HANDLED) error	code	invalid_type
message	Invalid type for 'prompt': expected one of a string or array of strings, integers, or integer arrays, but got an object instead.
param	prompt
type	invalid_request_error
  in sub openai-request at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/2BF7550AA561E7F663064FDF82CE819D1E6B76E0 (WWW::OpenAI::Request) line 224
  in sub OpenAITextCompletion at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/535D81CE175E38C47ED451F7D25E56DA34AF9CFC (WWW::OpenAI::TextCompletions) line 191
  in sub OpenAITextCompletion at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/535D81CE175E38C47ED451F7D25E56DA34AF9CFC (WWW::OpenAI::TextCompletions) line 58
  in method eval at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/A3A7BBA491E5C9A0E749FC3AAF2CD5DF578099ED (LLM::Functions::Evaluator) line 119
  in method eval at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/E9958E936A0343A821A4826F51C09FBEA301011B (LLM::Functions::Chat) line 87
  in method eval at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/E9958E936A0343A821A4826F51C09FBEA301011B (LLM::Functions::Chat) line 55
  in block <unit> at <unknown file> line 1
  in method eval at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/0754FDBAE95B47119FF01AC0B65A28EA4F198BA6 (Jupyter::Chatbook::Sandbox) line 155
  in code  at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/89001669598C3895746BA4CCE2D0F7B1A97AC115 (Jupyter::Chatbook) line 147
 (HANDLED) error	code	invalid_type
message	Invalid type for 'prompt': expected one of a string or array of strings, integers, or integer arrays, but got an object instead.
param	prompt
type	invalid_request_error
  in sub openai-request at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/2BF7550AA561E7F663064FDF82CE819D1E6B76E0 (WWW::OpenAI::Request) line 224
  in sub OpenAITextCompletion at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/535D81CE175E38C47ED451F7D25E56DA34AF9CFC (WWW::OpenAI::TextCompletions) line 191
  in sub OpenAITextCompletion at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/535D81CE175E38C47ED451F7D25E56DA34AF9CFC (WWW::OpenAI::TextCompletions) line 58
  in method eval at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/A3A7BBA491E5C9A0E749FC3AAF2CD5DF578099ED (LLM::Functions::Evaluator) line 119
  in method eval at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/E9958E936A0343A821A4826F51C09FBEA301011B (LLM::Functions::Chat) line 87
  in method eval at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/E9958E936A0343A821A4826F51C09FBEA301011B (LLM::Functions::Chat) line 55
  in block <unit> at <unknown file> line 1
  in method eval at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/0754FDBAE95B47119FF01AC0B65A28EA4F198BA6 (Jupyter::Chatbook::Sandbox) line 155
  in code  at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/89001669598C3895746BA4CCE2D0F7B1A97AC115 (Jupyter::Chatbook) line 147
]

In [13]:
%% markdown
$chatObj.eval('How do you make the original word from the characters in the previous example')

[(HANDLED) error	code	invalid_type
message	Invalid type for 'prompt': expected one of a string or array of strings, integers, or integer arrays, but got an object instead.
param	prompt
type	invalid_request_error
  in sub openai-request at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/2BF7550AA561E7F663064FDF82CE819D1E6B76E0 (WWW::OpenAI::Request) line 224
  in sub OpenAITextCompletion at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/535D81CE175E38C47ED451F7D25E56DA34AF9CFC (WWW::OpenAI::TextCompletions) line 191
  in sub OpenAITextCompletion at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/535D81CE175E38C47ED451F7D25E56DA34AF9CFC (WWW::OpenAI::TextCompletions) line 58
  in method eval at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/A3A7BBA491E5C9A0E749FC3AAF2CD5DF578099ED (LLM::Functions::Evaluator) line 119
  in method eval at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/E9958E936A0343A821A4826F51C09FBEA301011B (LLM::Functions::Chat) line 87
  in method eval at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/E9958E936A0343A821A4826F51C09FBEA301011B (LLM::Functions::Chat) line 55
  in block <unit> at <unknown file> line 1
  in method eval at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/0754FDBAE95B47119FF01AC0B65A28EA4F198BA6 (Jupyter::Chatbook::Sandbox) line 155
  in code  at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/89001669598C3895746BA4CCE2D0F7B1A97AC115 (Jupyter::Chatbook) line 147
 (HANDLED) error	code	invalid_type
message	Invalid type for 'prompt': expected one of a string or array of strings, integers, or integer arrays, but got an object instead.
param	prompt
type	invalid_request_error
  in sub openai-request at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/2BF7550AA561E7F663064FDF82CE819D1E6B76E0 (WWW::OpenAI::Request) line 224
  in sub OpenAITextCompletion at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/535D81CE175E38C47ED451F7D25E56DA34AF9CFC (WWW::OpenAI::TextCompletions) line 191
  in sub OpenAITextCompletion at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/535D81CE175E38C47ED451F7D25E56DA34AF9CFC (WWW::OpenAI::TextCompletions) line 58
  in method eval at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/A3A7BBA491E5C9A0E749FC3AAF2CD5DF578099ED (LLM::Functions::Evaluator) line 119
  in method eval at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/E9958E936A0343A821A4826F51C09FBEA301011B (LLM::Functions::Chat) line 87
  in method eval at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/E9958E936A0343A821A4826F51C09FBEA301011B (LLM::Functions::Chat) line 55
  in block <unit> at <unknown file> line 1
  in method eval at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/0754FDBAE95B47119FF01AC0B65A28EA4F198BA6 (Jupyter::Chatbook::Sandbox) line 155
  in code  at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/89001669598C3895746BA4CCE2D0F7B1A97AC115 (Jupyter::Chatbook) line 147
 (HANDLED) error	code	invalid_type
message	Invalid type for 'prompt': expected one of a string or array of strings, integers, or integer arrays, but got an object instead.
param	prompt
type	invalid_request_error
  in sub openai-request at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/2BF7550AA561E7F663064FDF82CE819D1E6B76E0 (WWW::OpenAI::Request) line 224
  in sub OpenAITextCompletion at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/535D81CE175E38C47ED451F7D25E56DA34AF9CFC (WWW::OpenAI::TextCompletions) line 191
  in sub OpenAITextCompletion at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/535D81CE175E38C47ED451F7D25E56DA34AF9CFC (WWW::OpenAI::TextCompletions) line 58
  in method eval at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/A3A7BBA491E5C9A0E749FC3AAF2CD5DF578099ED (LLM::Functions::Evaluator) line 119
  in method eval at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/E9958E936A0343A821A4826F51C09FBEA301011B (LLM::Functions::Chat) line 87
  in method eval at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/E9958E936A0343A821A4826F51C09FBEA301011B (LLM::Functions::Chat) line 55
  in block <unit> at <unknown file> line 1
  in method eval at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/0754FDBAE95B47119FF01AC0B65A28EA4F198BA6 (Jupyter::Chatbook::Sandbox) line 155
  in code  at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/89001669598C3895746BA4CCE2D0F7B1A97AC115 (Jupyter::Chatbook) line 147
 (HANDLED) error	code	invalid_type
message	Invalid type for 'prompt': expected one of a string or array of strings, integers, or integer arrays, but got an object instead.
param	prompt
type	invalid_request_error
  in sub openai-request at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/2BF7550AA561E7F663064FDF82CE819D1E6B76E0 (WWW::OpenAI::Request) line 224
  in sub OpenAITextCompletion at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/535D81CE175E38C47ED451F7D25E56DA34AF9CFC (WWW::OpenAI::TextCompletions) line 191
  in sub OpenAITextCompletion at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/535D81CE175E38C47ED451F7D25E56DA34AF9CFC (WWW::OpenAI::TextCompletions) line 58
  in method eval at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/A3A7BBA491E5C9A0E749FC3AAF2CD5DF578099ED (LLM::Functions::Evaluator) line 119
  in method eval at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/E9958E936A0343A821A4826F51C09FBEA301011B (LLM::Functions::Chat) line 87
  in method eval at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/E9958E936A0343A821A4826F51C09FBEA301011B (LLM::Functions::Chat) line 55
  in block <unit> at <unknown file> line 1
  in method eval at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/0754FDBAE95B47119FF01AC0B65A28EA4F198BA6 (Jupyter::Chatbook::Sandbox) line 155
  in code  at /Users/stephenroe/.rakubrew/versions/moar-2025.08/share/perl6/site/sources/89001669598C3895746BA4CCE2D0F7B1A97AC115 (Jupyter::Chatbook) line 147
]

-----

## References

### Articles

[AA1] Anton Antonov,
["Workflows with LLM functions"](https://rakuforprediction.wordpress.com/2023/08/01/workflows-with-llm-functions/),
(2023),
[RakuForPrediction at WordPress](https://rakuforprediction.wordpress.com).

[AA2] Anton Antonov,
["Number guessing games: PaLM vs ChatGPT"](https://rakuforprediction.wordpress.com/2023/08/06/number-guessing-games-palm-vs-chatgpt/),
(2023),
[RakuForPrediction at WordPress](https://rakuforprediction.wordpress.com).

### Packages

[AAp1] Anton Antonov,
[Jupyter::Chatbook Raku package](https://github.com/antononcube/Raku-Jupyter-Chatbook),
(2023),
[GitHub/antononcube](https://github.com/antononcube).

[AAp2] Anton Antonov,
[LLM::Functions Raku package](https://github.com/antononcube/Raku-LLM-Functions),
(2023),
[GitHub/antononcube](https://github.com/antononcube).

[AAp3] Anton Antonov,
[WWW::OpenAI Raku package](https://github.com/antononcube/Raku-WWW-OpenAI),
(2023),
[GitHub/antononcube](https://github.com/antononcube).

[AAp4] Anton Antonov,
[WWW::PaLM Raku package](https://github.com/antononcube/Raku-WWW-PaLM),
(2023),
[GitHub/antononcube](https://github.com/antononcube).

[AAp5] Anton Antonov,
[Clipboard Raku package](https://github.com/antononcube/Raku-Clipboard),
(2023),
[GitHub/antononcube](https://github.com/antononcube).

[AAp6] Anton Antonov,
[Data::Translators Raku package](https://github.com/antononcube/Raku-Data-Translators),
(2023),
[GitHub/antononcube](https://github.com/antononcube).

[AAp7] Anton Antonov,
[Data::TypeSystem Raku package](https://github.com/antononcube/Raku-Data-TypeSystem),
(2023),
[GitHub/antononcube](https://github.com/antononcube).

[AAp8] Anton Antonov,
[Text::Plot Raku package](https://github.com/antononcube/Raku-Text-Plot),
(2022),
[GitHub/antononcube](https://github.com/antononcube).

[AAp9] Anton Antonov,
[Text::SubParsers Raku package](https://github.com/antononcube/Raku-Text-SubParsers),
(2023),
[GitHub/antononcube](https://github.com/antononcube).

[BDp1] Brian Duggan,
[Jupyter:Kernel Raku package](https://github.com/bduggan/raku-jupyter-kernel),
(2017-2023),
[GitHub/bduggan](https://github.com/bduggan).

[MSp1] Matthew Stuckwisch,
[Intl::Token::Number Raku package](https://github.com/alabamenhu/IntlTokenNumber)
(2021-2023),
[GitHub/alabamenhu](https://github.com/alabamenhu).